In [1]:
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline
%load_ext autoreload
%autoreload 2

import ctypes
try:
    ctypes.CDLL("/usr/lib64/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
    print("\u2713 Preloaded libcuda.so.1 from /usr/lib64")
except Exception as e:
    print(f"\u26a0 Could not preload libcuda.so.1: {e}")

if "CUDA_PATH" not in os.environ:
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True, executable='/bin/bash', capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"\u2713 Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            raise RuntimeError("CUDA_PATH not set.")
    except Exception as e:
        raise RuntimeError(f"Failed to load CUDA: {e}") from e

cuda_path = os.environ.get('CUDA_PATH')
if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld.split(':') if current_ld else []
    for p in cuda_lib_paths:
        if os.path.exists(p) and p not in ld_paths:
            ld_paths.insert(0, p)
    os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
    for p in cuda_lib_paths:
        if not os.path.exists(p):
            continue
        for name in ['libcudart.so', 'libcudart.so.12', 'libnvrtc.so.12']:
            path = os.path.join(p, name)
            if os.path.exists(path):
                try:
                    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                    print(f"\u2713 Preloaded {name}")
                except Exception as e:
                    print(f"\u26a0 Could not preload {name}: {e}")
                break

from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Mapping
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from src.belief_quantized.value_iteration import ValueIteration
import json
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')
print("\u2713 All imports successful")

if is_cupy:
    import cupy as cp
    print(f"Using backend: CuPy (GPU)  |  CuPy {cp.__version__}, devices: {cp.cuda.runtime.getDeviceCount()}")
else:
    print("Using backend: NumPy (CPU)")

"""
Mapping value function vs M (belief quantization level).
Fixed: n=2 (state quantization), LIDAR sensor, toy2 environment.
Sweep: M in [2, 3, 4, 5, 6]  ->  cardinalities [136, 816, 3876, 15504, 54264].
Progress is saved after each M so a failed/OOM run does not lose prior results.
"""

CWD: /global/home/hpc5656/SLAM
✓ Preloaded libcuda.so.1 from /usr/lib64
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Preloaded libcudart.so
✓ Preloaded libcudart.so
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)  |  CuPy 14.0.0rc1, devices: 8


'\nMapping value function vs M (belief quantization level).\nFixed: n=2 (state quantization), LIDAR sensor, toy2 environment.\nSweep: M in [2, 3, 4, 5, 6]  ->  cardinalities [136, 816, 3876, 15504, 54264].\nProgress is saved after each M so a failed/OOM run does not lose prior results.\n'

In [2]:
# -----------------------------------------------------------------------
# Shared configuration
# -----------------------------------------------------------------------
n = 2           # State quantization level (fixed)
M_list = [2, 3, 4, 5, 6]   # 5 belief-quantization points
beta    = 0.95
epsilon = 1e-6

# Batch size for p_n_M computation.
#   None  -> process all states at once (fastest, may OOM for large M)
#   int   -> process in chunks (e.g. j_batch_size = 500 for M >= 5)
# Change per-cell below if you hit OOM on larger M.
j_batch_size = None

initial_belief_index = 0

obstacles, area = load_obstacles_config(environment='toy2')
motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = LIDAR(fov=360, r_max=10.0, B=8)

out_dir = PROJECT_ROOT / 'notebooks' / 'mapping' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'value_sweep_results_MAP.json'

if results_path.exists():
    with open(results_path, 'r') as f:
        results = json.load(f)
    print(f"Loaded {len(results)} result(s) from {results_path.name}")
else:
    results = []
    print("No previous results; starting fresh.")

# Expected cardinalities for reference
from math import comb
N_n = n ** 4  # number of grid cells for n=2 -> 16 map cells, 2^16 possible maps but BQ uses M-simplices
print(f"\nSweep: n={n} (fixed), M = {M_list}")
print(f"β = {beta}, \u03b5 = {epsilon}, j_batch_size = {j_batch_size}")
print(f"Environment: toy2, area = {area}")
print(f"\nExpected |\u03a0_n^M| cardinalities (M-simplex over N_n={2**n**2} maps):")
for M in M_list:
    card = comb(2**n**2 + M - 1, M)
    print(f"  M={M}: {card:,}")

Loaded 4 result(s) from value_sweep_results_MAP.json

Sweep: n=2 (fixed), M = [2, 3, 4, 5, 6]
β = 0.95, ε = 1e-06, j_batch_size = None
Environment: toy2, area = (0, 10, 0, 10)

Expected |Π_n^M| cardinalities (M-simplex over N_n=16 maps):
  M=2: 136
  M=3: 816
  M=4: 3,876
  M=5: 15,504
  M=6: 54,264


In [3]:
# Level 1/5: M = 2
M = 2
if any(r.get('M') == M for r in results):
    print(f"Already have M={M}; skipping.")
else:
    # j_batch_size = None  # override here if needed
    try:
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3],
            quantization_level=n)
        bmdp = BeliefMDP_n_M_Mapping(
            M=M, β=beta, n=n,
            motion_model=motion_model, measurement_model=sensor,
            obstacles=obstacles, _map=grid_map,
            sigma_v=1.0,
            j_batch_size=j_batch_size, i_batch_size=1)
        bmdp.map.seed_from_obstacles(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else __import__('numpy').array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(V.mean())
        results.append({
            'M': M, 'n': n,
            'm_n': int(bmdp.SQ.m_n),
            'cardinality': int(bmdp.BQ.cardinality),
            'value_initial': v0,
            'value_mean': float(V.mean()),
            'iterations': int(vi.iteration_count),
        })
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved M={M}; {len(results)} total.")
    except Exception as e:
        print(f"Error M={M}: {e}")
        raise

Already have M=2; skipping.


In [4]:
# Level 2/5: M = 3
M = 3
if any(r.get('M') == M for r in results):
    print(f"Already have M={M}; skipping.")
else:
    # j_batch_size = None  # override here if needed
    try:
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3],
            quantization_level=n)
        bmdp = BeliefMDP_n_M_Mapping(
            M=M, β=beta, n=n,
            motion_model=motion_model, measurement_model=sensor,
            obstacles=obstacles, _map=grid_map,
            sigma_v=1.0,
            j_batch_size=j_batch_size, i_batch_size=128)
        bmdp.map.seed_from_obstacles(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else __import__('numpy').array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(V.mean())
        results.append({
            'M': M, 'n': n,
            'm_n': int(bmdp.SQ.m_n),
            'cardinality': int(bmdp.BQ.cardinality),
            'value_initial': v0,
            'value_mean': float(V.mean()),
            'iterations': int(vi.iteration_count),
        })
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved M={M}; {len(results)} total.")
    except Exception as e:
        print(f"Error M={M}: {e}")
        raise

Already have M=3; skipping.


In [5]:
# Level 3/5: M = 4  (cardinality ~3,876)
# If you hit OOM computing p_n_M, uncomment and adjust:
# j_batch_size = 500
i_batch_size=256
M = 4
if any(r.get('M') == M for r in results):
    print(f"Already have M={M}; skipping.")
else:
    try:
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3],
            quantization_level=n)
        bmdp = BeliefMDP_n_M_Mapping(
            M=M, β=beta, n=n,
            motion_model=motion_model, measurement_model=sensor,
            obstacles=obstacles, _map=grid_map,
            sigma_v=1.0,
            j_batch_size=j_batch_size, i_batch_size=i_batch_size)
        bmdp.map.seed_from_obstacles(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else __import__('numpy').array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(V.mean())
        results.append({
            'M': M, 'n': n,
            'm_n': int(bmdp.SQ.m_n),
            'cardinality': int(bmdp.BQ.cardinality),
            'value_initial': v0,
            'value_mean': float(V.mean()),
            'iterations': int(vi.iteration_count),
        })
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved M={M}; {len(results)} total.")
    except Exception as e:
        print(f"Error M={M}: {e}")
        raise

Already have M=4; skipping.


In [6]:
# Level 4/5: M = 5  (cardinality ~15,504)
# Recommend batching p_n_M if you hit OOM:
# j_batch_size = 200
M = 5
if any(r.get('M') == M for r in results):
    print(f"Already have M={M}; skipping.")
else:
    try:
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3],
            quantization_level=n)
        bmdp = BeliefMDP_n_M_Mapping(
            M=M, β=beta, n=n,
            motion_model=motion_model, measurement_model=sensor,
            obstacles=obstacles, _map=grid_map,
            sigma_v=1.0,
            j_batch_size=j_batch_size, i_batch_size=196)
        bmdp.map.seed_from_obstacles(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else __import__('numpy').array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(V.mean())
        results.append({
            'M': M, 'n': n,
            'm_n': int(bmdp.SQ.m_n),
            'cardinality': int(bmdp.BQ.cardinality),
            'value_initial': v0,
            'value_mean': float(V.mean()),
            'iterations': int(vi.iteration_count),
        })
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved M={M}; {len(results)} total.")
    except Exception as e:
        print(f"Error M={M}: {e}")
        raise

Already have M=5; skipping.


In [7]:
# Level 5/5: M = 6  (cardinality ~54,264)
# Likely needs batched p_n_M computation to avoid OOM:
# j_batch_size = 100
M = 6
if any(r.get('M') == M for r in results):
    print(f"Already have M={M}; skipping.")
else:
    try:
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3],
            quantization_level=n)
        bmdp = BeliefMDP_n_M_Mapping(
            M=M, β=beta, n=n,
            motion_model=motion_model, measurement_model=sensor,
            obstacles=obstacles, _map=grid_map,
            sigma_v=1.0,
            j_batch_size=j_batch_size, i_batch_size=72)
        bmdp.map.seed_from_obstacles(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else __import__('numpy').array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(V.mean())
        results.append({
            'M': M, 'n': n,
            'm_n': int(bmdp.SQ.m_n),
            'cardinality': int(bmdp.BQ.cardinality),
            'value_initial': v0,
            'value_mean': float(V.mean()),
            'iterations': int(vi.iteration_count),
        })
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved M={M}; {len(results)} total.")
    except Exception as e:
        print(f"Error M={M}: {e}")
        raise

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz
  Checking cache file: Q_n_n2_obs2_map2x2_B8_7a9a0a58.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map2x2_B8_7a9a0a58.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M6_N16.npz
Cache file does not exist: /global/home/hpc5656/SLAM/cache/MAP/p_n_M_mapping/p_n_M_mapping_M6_n2_map2x2_max2.0_f8105fc5.npz
Computing p_n_M for the first time...
This will compute: 4 actions × 54264² belief-to-belief transitions
  Note: Uses quantized-observation η_n_batch (no MC sampling)
Computing p_n_M for mapping (quantized η_n_batch, sparse)...
  States: 868,224 = m_n=16 × cardinality=54264, n_u=4
  Computing p_n_M for action 1/4...


Action 1/4:   0%|          | 0/868224 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Plot: Optimal cost vs M (belief quantization level)
valid = [r for r in results if r.get('value_initial') is not None]
if not valid:
    print("No valid results to plot yet.")
else:
    import numpy as onp
    pts = sorted(valid, key=lambda r: r['M'])
    xs = [r['M'] for r in pts]
    ys_init = [r['value_initial'] for r in pts]
    ys_mean = [r['value_mean'] for r in pts]
    cards   = [r['cardinality'] for r in pts]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Left: value vs M
    ax = axes[0]
    ax.plot(xs, ys_init, 'D-', color='#2980b9', markersize=8, label='V(initial belief)')
    ax.plot(xs, ys_mean, 's--', color='#e67e22', markersize=7, label='V (mean over beliefs)')
    ax.set_xlabel('Belief quantization level M')
    ax.set_ylabel('Optimal cost (value)')
    ax.set_title(f'Mapping: Optimal cost vs M  (n={n} fixed)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: value vs cardinality (log scale)
    ax = axes[1]
    ax.semilogx(cards, ys_init, 'D-', color='#2980b9', markersize=8, label='V(initial belief)')
    ax.semilogx(cards, ys_mean, 's--', color='#e67e22', markersize=7, label='V (mean over beliefs)')
    for x, y, M_val in zip(cards, ys_init, xs):
        ax.annotate(f'M={M_val}', (x, y), textcoords='offset points', xytext=(5, 4), fontsize=8)
    ax.set_xlabel(r'$|\Pi_n^M|$ (cardinality, log scale)')
    ax.set_ylabel('Optimal cost (value)')
    ax.set_title('Value vs belief-space cardinality')
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_dir / f'value_vs_M_MAP_n{n}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved figure to {out_dir / f'value_vs_M_MAP_n{n}.png'}")

In [ ]:
# Summary table
valid = [r for r in results if r.get('value_initial') is not None]
if valid:
    print(f"{'M':>4} {'m_n':>6} {'cardinality':>14} {'V(init)':>10} {'V(mean)':>10} {'iters':>6}")
    print('-' * 56)
    for r in sorted(valid, key=lambda x: x['M']):
        print(f"{r['M']:>4} {r.get('m_n', 0):>6} {r.get('cardinality', 0):>14,} "
              f"{r['value_initial']:>10.4f} {r.get('value_mean', 0):>10.4f} {r.get('iterations', 0):>6}")
else:
    print("No valid results yet.")